In [2]:
from transformers import AutoTokenizer,BitsAndBytesConfig, AutoModelForCausalLM
from peft import PeftModel
import torch
import pandas as pd
import re
from tqdm.notebook import tqdm
from prepare_data_for_dpo import extract_features_from_answer,generate_responses_batched,is_valid_response,process_responses_list
from prepare_data_for_prompts_classifier import detect_english_text
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
bnb_config_base_model=BitsAndBytesConfig(

    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN')
model_name = "Qwen/Qwen3-4B"
tokenizer=AutoTokenizer.from_pretrained(model_name,token=HF_TOKEN)
tokenizer.pad_token=tokenizer.eos_token

In [ ]:
tokenizer=AutoTokenizer.from_pretrained('sft_model')
base_model_for_sft=AutoModelForCausalLM.from_pretrained(model_name,
                                                         device_map='auto',
                                                         quantization_config=bnb_config_base_model,
                                                         dtype=torch.float16,
                                                         token=HF_TOKEN
                                                        )
sft_model=PeftModel.from_pretrained(base_model_for_sft,'sft_model')
sft_model.eval()

In [ ]:
test_data=pd.read_csv('test_data_from_sft_data1')
test_data=extract_features_from_answer(test_data)
test_data_sample=test_data.sample(400)
test_data_sample_prompts=test_data_sample['user_message'].tolist()
test_data_sample_labels=test_data_sample['is_unsafe'].tolist()

In [ ]:
results=generate_responses_batched(sft_model,tokenizer,test_data_sample_prompts,test_data_sample_labels)

In [ ]:
results=pd.DataFrame(results)

In [3]:
results=pd.read_csv('answers_from_sft_model.csv',index_col=0)
results['responses'] = results['responses'].apply(eval)

In [4]:
results['responses'] = results['responses'].apply(lambda responses:[r for r in responses if  detect_english_text(r,min_confidence=0.95)] )

results=results[results['responses'].apply(len)>=2]

In [7]:
results.info()


<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   prompt         400 non-null    str   
 1   label          400 non-null    int64 
 2   responses      400 non-null    object
 3   response_part  400 non-null    object
 4   analyses_part  400 non-null    object
dtypes: int64(1), object(3), str(1)
memory usage: 112.4+ KB


In [6]:
results['response_part'], results['analyses_part'] = zip(*results['responses'].apply(process_responses_list))

In [11]:
results['responses_with_prompt'] = results.apply(
    lambda row: ['Prompt: '+row['prompt'] + '\nAnswer: ' + response for response in row['response_part']],
    axis=1
)

In [12]:
print(results['responses_with_prompt'].iloc[7])

["Prompt: In the digital age, teams are often geographically dispersed and may have members from different generations with varying levels of comfort with technology. What is the most effective way of leading a team in this environment?\nAnswer: I think the best approach is to focus on the team's goals and ensure that all members are aligned with the team's purpose. This includes setting clear expectations, encouraging collaboration, and providing opportunities for everyone to contribute. Additionally, it's important to create an inclusive environment that respects the diverse backgrounds and experiences of all team members. I think this approach can help create a sense of unity and purpose that can lead to better performance and satisfaction for all members.", 'Prompt: In the digital age, teams are often geographically dispersed and may have members from different generations with varying levels of comfort with technology. What is the most effective way of leading a team in this envir